# Exercise 2: Convolutional Neural Network on CIFAR-10

**Aim.** Replace the Exercise 1 MLP with a CNN containing exactly three trainable convolutional layers. The same seeded 45,000/5,000 train/validation split, normalisation statistics, Adam optimiser, batch size, loss, and 30 training epochs are used so that the MLP and CNN can be compared fairly. The original 10,000-image test partition is used only for final evaluation.

**How to run.** Install `requirements.txt`, start Jupyter, and use **Kernel > Restart Kernel and Run All**.

## 1. Setup: imports, reproducibility, and paths

In [ ]:
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
for directory in (DATA_DIR, OUTPUT_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = OUTPUT_DIR / "exercise1 config.json"
MLP_METRICS_PATH = OUTPUT_DIR / "exercise1 metrics.json"
MLP_HISTORY_PATH = OUTPUT_DIR / "exercise1 history.csv"

with CONFIG_PATH.open("r") as config_file:
    exercise1_config = json.load(config_file)

SEED = exercise1_config["seed"]
BATCH_SIZE = exercise1_config["batch_size"]
EPOCHS = exercise1_config["epochs"]
NUM_WORKERS = exercise1_config["num_workers"]
CHANNEL_MEAN = exercise1_config["normalisation"]["mean"]
CHANNEL_STD = exercise1_config["normalisation"]["std"]
VAL_INDICES = exercise1_config["validation_indices"]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 2. Dataset split and preprocessing

CIFAR-10 contains 50,000 training images and 10,000 test images. Exercise 1 used the validation indices from a seeded 45,000/5,000 random split. Exercise 2 recreates that same split exactly by loading the full training dataset and selecting those indices. Channel means and standard deviations are also reused.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CHANNEL_MEAN, CHANNEL_STD),
])

try:
    train_full_dataset = datasets.CIFAR10(DATA_DIR, train=True, download=True, transform=transform)
    test_dataset = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=transform)
except Exception as error:
    raise RuntimeError(
        f"Could not download or load CIFAR-10 in '{DATA_DIR}'."
    ) from error

ALL_TRAIN_INDICES = list(range(50_000))
train_indices = [index for index in ALL_TRAIN_INDICES if index not in set(VAL_INDICES)]
train_subset = Subset(train_full_dataset, train_indices)
val_subset = Subset(train_full_dataset, VAL_INDICES)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                        persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                         persistent_workers=NUM_WORKERS > 0)

CLASS_NAMES = train_full_dataset.classes
print(f"Training images: {len(train_subset):,}")
print(f"Validation images: {len(val_subset):,}")
print(f"Test images: {len(test_dataset):,}")

## 3. CNN architecture

The model has exactly three trainable convolutional layers:

1. `Conv2d(3, 32, 3)` learns local RGB patterns.
2. `Conv2d(32, 64, 3)` combines the first-layer feature maps.
3. `Conv2d(64, 10, 3)` produces one feature map per CIFAR-10 class.

ReLU activations and max pooling provide non-linearity and spatial reduction. Adaptive average pooling converts each class map to a 1-element logit, so no additional fully connected trainable layer is needed.

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(64, 10, kernel_size=3, padding=1),
+            nn.ReLU(),
+            nn.AdaptiveAvgPool2d((1, 1)),
+        )
+
+    def forward(self, images):
+        logits = self.features(images)
+        return logits.flatten(1)
+
+
+def count_trainable_parameters(model):
+    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
+
+
+model = CNN().to(DEVICE)
+PARAMETER_COUNT = count_trainable_parameters(model)
+print(model)
+print(f"Trainable parameters: {PARAMETER_COUNT:,}")

## 4. Training and evaluation functions

Cross entropy is used for ten mutually exclusive classes. Epoch losses are sample-weighted, and the model saving the lowest validation loss is retained. The test set will be evaluated only after the main run.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
+    model.train()
+    total_loss, total_correct, total_samples = 0.0, 0, 0
+    for images, labels in loader:
+        images = images.to(DEVICE, non_blocking=True)
+        labels = labels.to(DEVICE, non_blocking=True)
+        optimizer.zero_grad()
+        logits = model(images)
+        loss = criterion(logits, labels)
+        loss.backward()
+        optimizer.step()
+        total_loss += loss.item() * labels.size(0)
+        total_correct += (logits.argmax(dim=1) == labels).sum().item()
+        total_samples += labels.size(0)
+    return total_loss / total_samples, 100 * total_correct / total_samples
+
+
+def evaluate(model, loader, criterion):
+    model.eval()
+    total_loss, total_samples = 0.0, 0
+    all_predictions, all_labels = [], []
+    with torch.no_grad():
+        for images, labels in loader:
+            images = images.to(DEVICE, non_blocking=True)
+            labels = labels.to(DEVICE, non_blocking=True)
+            logits = model(images)
+            total_loss += criterion(logits, labels).item() * labels.size(0)
+            total_samples += labels.size(0)
+            all_predictions.append(logits.argmax(dim=1).cpu())
+            all_labels.append(labels.cpu())
+    predictions = torch.cat(all_predictions).numpy()
+    labels = torch.cat(all_labels).numpy()
+    accuracy = 100 * (predictions == labels).mean()
+    return total_loss / total_samples, accuracy, predictions, labels
+
+
+def train_model(model, optimizer, criterion, epochs, checkpoint_path):
+    history = []
+    best_val_loss = float("inf")
+    best_epoch = 0
+    for epoch in range(1, epochs + 1):
+        epoch_start = time.perf_counter()
+        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer)
+        val_loss, val_accuracy, _, _ = evaluate(model, val_loader, criterion)
+        epoch_seconds = time.perf_counter() - epoch_start
+        improved = val_loss < best_val_loss
+        if improved:
+            best_val_loss = val_loss
+            best_epoch = epoch
+            torch.save(model.state_dict(), checkpoint_path)
+        history.append({
+            "epoch": epoch,
+            "train_loss": train_loss,
+            "train_accuracy": train_accuracy,
+            "val_loss": val_loss,
+            "val_accuracy": val_accuracy,
+            "epoch_seconds": epoch_seconds,
+        })
+        print(f"epoch {epoch:2d}/{epochs} | train loss {train_loss:.4f} acc {train_accuracy:5.2f}% | "
+              f"val loss {val_loss:.4f} acc {val_accuracy:5.2f}% | {epoch_seconds:5.1f}s"
+              f"{' *' if improved else ''}")
+    return pd.DataFrame(history), best_epoch

## 5. Main training run

Exercise 1 selected Adam with learning rate `1e-4`. The same choice is used here for comparable optimisation. A fresh CNN starts from a seeded random initialisation.

In [ ]:
set_seed(SEED)
+model = CNN().to(DEVICE)
+criterion = nn.CrossEntropyLoss()
+optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
+CHECKPOINT_PATH = OUTPUT_DIR / "exercise2 best model.pt"
+HISTORY_PATH = OUTPUT_DIR / "exercise2 history.csv"
+
+history, best_epoch = train_model(model, optimizer, criterion, EPOCHS, CHECKPOINT_PATH)
+history.to_csv(HISTORY_PATH, index=False)
+model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
+
+print(f"Best validation epoch: {best_epoch}")

## 6. Final train, validation, and test evaluation

In [ ]:
train_loss, train_accuracy, _, _ = evaluate(model, train_loader, criterion)
+val_loss, val_accuracy, val_predictions, val_labels = evaluate(model, val_loader, criterion)
+test_loss, test_accuracy, test_predictions, test_labels = evaluate(model, test_loader, criterion)
+
+metrics = {
+    "train": {"loss": float(train_loss), "accuracy": float(train_accuracy)},
+    "validation": {"loss": float(val_loss), "accuracy": float(val_accuracy)},
+    "test": {"loss": float(test_loss), "accuracy": float(test_accuracy)},
+    "train_minus_test_accuracy_gap": float(train_accuracy - test_accuracy),
+    "best_epoch": int(best_epoch),
+    "parameter_count": int(PARAMETER_COUNT),
+    "optimiser": "Adam lr=1e-4",
+    "device": str(DEVICE),
+}
+
+METRICS_PATH = OUTPUT_DIR / "exercise2 metrics.json"
+with METRICS_PATH.open("w") as metrics_file:
+    json.dump(metrics, metrics_file, indent=2)
+
+print(classification_report(test_labels, test_predictions, target_names=CLASS_NAMES, digits=4))
+print(pd.DataFrame([
+    {"partition": "Training", "loss": train_loss, "accuracy": train_accuracy},
+    {"partition": "Validation", "loss": val_loss, "accuracy": val_accuracy},
+    {"partition": "Test", "loss": test_loss, "accuracy": test_accuracy},
+]).round(4).to_string(index=False))

## 7. Loss and accuracy curves

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
+axes[0].plot(history["epoch"], history["train_loss"], label="Training")
+axes[0].plot(history["epoch"], history["val_loss"], label="Validation")
+axes[0].axvline(best_epoch, color="gray", linestyle="--", label=f"Best epoch {best_epoch}")
+axes[0].set(title="CNN loss", xlabel="Epoch", ylabel="Cross-entropy loss")
+axes[0].grid(alpha=0.25)
+axes[0].legend()
+
+axes[1].plot(history["epoch"], history["train_accuracy"], label="Training")
+axes[1].plot(history["epoch"], history["val_accuracy"], label="Validation")
+axes[1].axvline(best_epoch, color="gray", linestyle="--", label=f"Best epoch {best_epoch}")
+axes[1].set(title="CNN accuracy", xlabel="Epoch", ylabel="Accuracy (%)")
+axes[1].grid(alpha=0.25)
+axes[1].legend()
+figure.tight_layout()
+figure.savefig(FIGURE_DIR / "exercise2 training curves.png", dpi=300, bbox_inches="tight")
+plt.show()

## 8. Test confusion matrix

In [ ]:
matrix = confusion_matrix(test_labels, test_predictions)
+figure, axis = plt.subplots(figsize=(9, 7))
+image = axis.imshow(matrix, cmap="Blues")
+axis.set_xticks(range(10), CLASS_NAMES, rotation=45, ha="right")
+axis.set_yticks(range(10), CLASS_NAMES)
+axis.set_xlabel("Predicted class")
+axis.set_ylabel("True class")
+axis.set_title("Exercise 2 CNN confusion matrix")
+for row in range(10):
+    for column in range(10):
+        axis.text(column, row, str(matrix[row, column]), ha="center", va="center", fontsize=7,
+                  color="white" if matrix[row, column] > matrix.max() / 2 else "black")
+figure.colorbar(image, ax=axis, fraction=0.046)
+figure.tight_layout()
+figure.savefig(FIGURE_DIR / "exercise2 confusion matrix.png", dpi=300, bbox_inches="tight")
+plt.show()

## 9. Direct MLP versus CNN comparison

The table below combines Exercise 1's saved metrics with the current CNN metrics. It compares test accuracy, training and test loss, generalisation gap, and trainable parameter count.

In [ ]:
with MLP_METRICS_PATH.open("r") as metrics_file:
+    mlp_metrics = json.load(metrics_file)
+
+comparison = pd.DataFrame([
+    {
+        "Model": "MLP",
+        "Test accuracy (%)": mlp_metrics["test"]["accuracy"],
+        "Training loss": mlp_metrics["train"]["loss"],
+        "Validation loss": mlp_metrics["validation"]["loss"],
+        "Test loss": mlp_metrics["test"]["loss"],
+        "Train-test accuracy gap": mlp_metrics["train_minus_test_accuracy_gap"],
+        "Trainable parameters": mlp_metrics["parameter_count"],
+    },
+    {
+        "Model": "CNN",
+        "Test accuracy (%)": test_accuracy,
+        "Training loss": train_loss,
+        "Validation loss": val_loss,
+        "Test loss": test_loss,
+        "Train-test accuracy gap": train_accuracy - test_accuracy,
+        "Trainable parameters": PARAMETER_COUNT,
+    },
+])
+comparison.round(4).to_csv(OUTPUT_DIR / "exercise1 vs exercise2 comparison.csv", index=False)
+display(comparison.round(4))
+
+figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
+axes[0].bar(comparison["Model"], comparison["Test accuracy (%)"], color=["#777777", "#4c78a8"])
+axes[0].set_ylabel("Test accuracy (%)")
+axes[0].set_title("Test accuracy comparison")
+axes[0].grid(axis="y", alpha=0.25)
+
+axes[1].bar(comparison["Model"], comparison["Trainable parameters"], color=["#777777", "#4c78a8"])
+axes[1].set_ylabel("Trainable parameters")
+axes[1].set_title("Parameter count comparison")
+axes[1].ticklabel_format(axis="y", style="plain")
+axes[1].grid(axis="y", alpha=0.25)
+figure.tight_layout()
+figure.savefig(FIGURE_DIR / "exercise1 vs exercise2 comparison.png", dpi=300, bbox_inches="tight")
+plt.show()

## 10. Results and interpretation

Convolutional layers learn spatially local features by applying shared filters across an image. Early filters commonly respond to edges and colour transitions, while deeper filters combine these patterns into textures and object parts. Parameter sharing means a CNN needs fewer parameters than a flattened MLP in this experiment, and weight sharing gives translation tolerance.

Use the printed comparison table and the saved plots when discussing the report. Compare the two models using test accuracy, training/validation/test loss, the train-test accuracy gap, and parameter count. Explain differences as evidence of spatial inductive bias rather than assuming that a CNN is always better for every dataset or setting.